In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pickle
import os
import glob
import shap
import json
import networkx as nx
import seaborn as sns
import lingam
import warnings

from sklearn.preprocessing import (
    LabelEncoder,
    StandardScaler,
    MinMaxScaler,
)

from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from typing import Dict
from sklearn.model_selection import cross_val_score, KFold
from sklearn.model_selection import train_test_split
from sklearn.model_selection import TimeSeriesSplit
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, f1_score
from scipy.stats import pearsonr
from statsmodels.tsa.stattools import ccf
from statsmodels.stats.stattools import durbin_watson
from statsmodels.tsa.api import VAR
from scipy import stats


from sklearn.metrics import (
    roc_auc_score, 
    roc_curve, 
    confusion_matrix, 
    precision_score, 
    recall_score, 
    accuracy_score,
    classification_report
)


warnings.filterwarnings('ignore')
print("✅ Imports loaded successfully!")



✅ Imports loaded successfully!


<br> <br> <br>

## Create Causal Graph 

Create a causal graph on machines dataset

In [2]:
def create_causal_network(df, features, regularize=True, noise_level=1e-4):
    X_selected = df[features].values

    # Initialize the VARLiNGAM model
    model = lingam.VARLiNGAM()

    # Attempt to fit the model; add noise if a LinAlgError occurs
    try:
        model.fit(X_selected)
    except np.linalg.LinAlgError as e:
        print("LinAlgError encountered during model fitting:", e)
        if regularize:
            print(f"Applying regularization: adding noise (std={noise_level}) to the data and trying again.")
            X_selected += np.random.normal(0, noise_level, X_selected.shape)
            model.fit(X_selected)
        else:
            raise e

    # Retrieve the adjacency matrices (one per lag)
    adjacency_matrices = model.adjacency_matrices_

    # Build a directed graph and create a list of dictionaries for each non-zero edge.
    G = nx.DiGraph()
    G.add_nodes_from(features)
    n_features = len(features)
    structured_adjacency = []

    # Loop through each adjacency matrix (one per lag)
    for lag_index, adj_matrix in enumerate(adjacency_matrices):
        for i in range(n_features):
            for j in range(n_features):
                weight = adj_matrix[i, j]
                if weight != 0:
                    edge_dict = {
                        "source_feature": features[i],
                        "target_feature": features[j],
                        "effect_strength": weight,
                    }
                    structured_adjacency.append(edge_dict)
                    G.add_edge(features[i], features[j], weight=weight)

    # Plot the graph using a spring layout for clarity
    pos = nx.spring_layout(G, seed=42)  # seed for reproducibility
    fig, ax = plt.subplots(figsize=(12, 8))
    nx.draw(
        G,
        pos,
        with_labels=True,
        node_color="lightblue",
        node_size=1500,
        arrowstyle="->",
        arrowsize=20,
        edge_color="gray",
        font_size=10,
        ax=ax,
    )

    # Display the edge weights formatted to two decimal places
    edge_labels = nx.get_edge_attributes(G, "weight")
    edge_labels = {edge: f"{weight:.2f}" for edge, weight in edge_labels.items()}
    nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels, font_color="red", ax=ax)

    ax.set_title("Causal Network Graph from VARLiNGAM")
    ax.axis("off")
    # plt.show()
    plt.close()

    return structured_adjacency, fig


In [3]:
machine_number = 1

In [4]:
df_machine = pd.read_csv(f"../../data/azure_pm/lag_features/machine_{machine_number}_lag_features.csv")
df_machine.drop(columns=["datetime", "failure", "comp"], inplace=True)

## Create causal graph
structured_adjacency, fig = create_causal_network(df=df_machine, features=df_machine.columns)

In [6]:
# Ensure the target directory exists
output_dir = f"../../causality_graph/azure_pm/machine_{machine_number}/"
os.makedirs(output_dir, exist_ok=True)

# Save structured_adjacency as JSON
with open(os.path.join(output_dir, "structured_adjacency.json"), "w") as f:
    json.dump(structured_adjacency, f, indent=2)

# Save the figure as PNG
fig.savefig(os.path.join(output_dir, "causal_network.png"))

<br> <br>

### Read Causal graph file

In [7]:
with open(f"../../causality_graph/azure_pm/machine_{machine_number}/structured_adjacency.json", "r") as f:
    structured_adjacency = json.load(f)

print(f"Loaded {len(structured_adjacency)} edges from structured_adjacency.json")

Loaded 667 edges from structured_adjacency.json


<br> <br> <br>

---